# 02. HiFi-GAN 학습

**실행 순서 (매번 동일)**
1. 셀 1~19 순서대로 실행
2. 셀 10 (wandb 초기화) — `name` 수정 후 실행
3. 셀 11 (체크포인트 로드) — 이어서 학습 시 실행, 처음부터면 건너뜀
4. 셀 12 (학습 루프) 실행
5. 셀 13 (git push) 실행

In [1]:
# 셀 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 셀 2. GitHub 클론
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR = '/content/seismic-gen'
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/isseul/cose362-k08-seismic-generation.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin hifigan-dev

!cd {REPO_DIR} && git checkout hifigan-dev
print('✅ 완료')

Cloning into '/content/seismic-gen'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 152 (delta 54), reused 114 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 6.11 MiB | 15.89 MiB/s, done.
Resolving deltas: 100% (54/54), done.
Branch 'hifigan-dev' set up to track remote branch 'hifigan-dev' from 'origin'.
Switched to a new branch 'hifigan-dev'
✅ 완료


In [ ]:
# 셀 3. 라이브러리 설치 & 기본 설정
!pip install torch torchaudio numpy matplotlib wandb -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os, sys, json, glob, time, itertools

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# 경로
SPEC_DIR = '/content/drive/MyDrive/ML_Project/outputs/spectrograms_finetune_m4'
HIGAN_SAVE = '/content/drive/MyDrive/ML_Project/outputs/hifigan_finetune_m4'
os.makedirs(HIGAN_SAVE, exist_ok=True)

# STFT 설정
SR         = 100
N_FFT      = 160
HOP_LENGTH = 46
WIN_LENGTH = 160
N_FREQ     = N_FFT // 2 + 1  # 81
N_SAMPLES  = 6000

# 실험 파라미터
BATCH_SIZE = 8
N_EPOCHS   = 200
LAMBDA_STFT = 45

print('✅ 설정 완료')

device: cuda
✅ 설정 완료


In [ ]:
# 셀 4. HiFi-GAN 원본 클론
if not os.path.exists('/content/hifi-gan'):
    !git clone https://github.com/jik876/hifi-gan.git /content/hifi-gan
sys.path.insert(0, '/content/hifi-gan')
print('✅ 완료')

✅ 완료


In [ ]:
# 셀 5. Config 저장
from env import AttrDict

config = {
    "resblock": "1",
    "num_gpus": 0,
    "learning_rate": 0.0002,
    "adam_b1": 0.8,
    "adam_b2": 0.99,
    "lr_decay": 0.999,
    "seed": 1234,

    # ── 원본 유지 ──────────────────────────
    "upsample_rates": [8, 8, 2, 2],
    "upsample_kernel_sizes": [16, 16, 4, 4],
    "upsample_initial_channel": 256,
    "resblock_kernel_sizes": [3, 7, 11],
    "resblock_dilation_sizes": [[1,3,5],[1,3,5],[1,3,5]],

    # ── 지진파 맞춤 수정 ───────────────────
    "num_mels": 81,
    "n_fft": 160,
    "hop_size": 46,
    "win_size": 160,
    "sampling_rate": 100,
    "segment_size": 6000,

    # ── 실험 파라미터 ──────────────────────
    "batch_size": 8,
    "num_workers": 2
}

os.makedirs('/content/seismic-gen/hifigan', exist_ok=True)
config_path = '/content/seismic-gen/hifigan/config_seismic.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)

h = AttrDict(config)
print('✅ config 저장 완료')

✅ config 저장 완료


In [ ]:
import os, shutil

LOCAL_SPEC_DIR = '/content/spectrograms_finetune_m4'
DRIVE_SPEC_DIR = '/content/drive/MyDrive/ML_Project/outputs/spectrograms_finetune_m4'

if not os.path.exists(LOCAL_SPEC_DIR):
    os.makedirs(LOCAL_SPEC_DIR, exist_ok=True)
    total = 0
    for split in ['train', 'val']:
        src_split = os.path.join(DRIVE_SPEC_DIR, split)
        dst_split = os.path.join(LOCAL_SPEC_DIR, split)
        os.makedirs(dst_split, exist_ok=True)
        files = os.listdir(src_split)
        for i, f in enumerate(files):
            shutil.copy(os.path.join(src_split, f), os.path.join(dst_split, f))
            total += 1
            if total % 1000 == 0:
                print(f'{total}개 복사 완료...')
    print(f'✅ 완료! 총 {total}개')
else:
    print('이미 존재, 스킵')

SPEC_DIR = LOCAL_SPEC_DIR

이미 존재, 스킵


In [ ]:
# 셀 6. Dataset & DataLoader
from torch.utils.data import Dataset, DataLoader

class SeismicDataset(Dataset):
    # VAE 추출 스펙트럼 + 원본 파형 로딩
    def __init__(self, spec_dir, split, segment_size=6000, hop_size=HOP_LENGTH):
        self.segment_size = segment_size
        self.hop_size     = hop_size
        all_recon = sorted(glob.glob(os.path.join(spec_dir, split, '*_recon.npy')))
        self.recon_files = [
            r for r in all_recon
            if os.path.exists(r[:-len('_recon.npy')] + '_wf.npy')
        ]
        print(f'[{split}] {len(self.recon_files):,}개 (누락 {len(all_recon)-len(self.recon_files)}개 제외)')

    def __len__(self):
        return len(self.recon_files)

    def __getitem__(self, idx):
        recon_path = self.recon_files[idx]
        wf_path    = recon_path[:-len('_recon.npy')] + '_wf.npy'
        recon = np.load(recon_path).astype(np.float32)
        wf    = np.load(wf_path).astype(np.float32)
        recon_t = torch.from_numpy(recon)
        wf_t    = torch.from_numpy(wf)
        if wf_t.shape[0] >= self.segment_size:
            start    = torch.randint(0, wf_t.shape[0] - self.segment_size + 1, (1,)).item()
            wf_seg   = wf_t[start:start + self.segment_size]
            start_f  = start // self.hop_size
            n_frames = self.segment_size // self.hop_size
            spec_seg = recon_t[:, start_f:start_f + n_frames]
            if spec_seg.shape[1] < n_frames:
                spec_seg = F.pad(spec_seg, (0, n_frames - spec_seg.shape[1]))
        else:
            wf_seg   = F.pad(wf_t, (0, self.segment_size - wf_t.shape[0]))
            spec_seg = recon_t
        return spec_seg, wf_seg.unsqueeze(0)

train_ds = SeismicDataset(SPEC_DIR, 'train')
val_ds   = SeismicDataset(SPEC_DIR, 'val')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
spec_b, wf_b = next(iter(train_loader))
print(f'spec : {spec_b.shape}')
print(f'wf   : {wf_b.shape}')

[train] 5,375개 (누락 0개 제외)
[val] 1,210개 (누락 0개 제외)
spec : torch.Size([8, 81, 130])
wf   : torch.Size([8, 1, 6000])


In [ ]:
# 셀 7. 모델 정의 (HiFi-GAN 원본 + 지진파 수정)
from utils import init_weights, get_padding
from torch.nn import Conv1d, ConvTranspose1d, AvgPool1d, Conv2d
from torch.nn.utils import weight_norm, remove_weight_norm, spectral_norm

LRELU_SLOPE = 0.1

# ── 원본 유지 ──────────────────────────────────────────────────
class ResBlock1(torch.nn.Module):
    def __init__(self, h, channels, kernel_size=3, dilation=(1, 3, 5)):
        super(ResBlock1, self).__init__()
        self.h = h
        self.convs1 = nn.ModuleList([
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[0], padding=get_padding(kernel_size, dilation[0]))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[1], padding=get_padding(kernel_size, dilation[1]))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=dilation[2], padding=get_padding(kernel_size, dilation[2])))
        ])
        self.convs1.apply(init_weights)
        self.convs2 = nn.ModuleList([
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1))),
            weight_norm(Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=get_padding(kernel_size, 1)))
        ])
        self.convs2.apply(init_weights)

    def forward(self, x):
        for c1, c2 in zip(self.convs1, self.convs2):
            xt = F.leaky_relu(x, LRELU_SLOPE)
            xt = c1(xt)
            xt = F.leaky_relu(xt, LRELU_SLOPE)
            xt = c2(xt)
            x  = xt + x
        return x

    def remove_weight_norm(self):
        for l in self.convs1: remove_weight_norm(l)
        for l in self.convs2: remove_weight_norm(l)


class Generator(torch.nn.Module):
    def __init__(self, h):
        super(Generator, self).__init__()
        self.h = h
        self.num_kernels   = len(h.resblock_kernel_sizes)
        self.num_upsamples = len(h.upsample_rates)
        # ── 수정: 입력 채널 80 → h.num_mels (81) ──────────────
        self.conv_pre = weight_norm(Conv1d(h.num_mels, h.upsample_initial_channel, 7, 1, padding=3))
        # ── 원본 유지 ──────────────────────────────────────────
        self.ups = nn.ModuleList()
        for i, (u, k) in enumerate(zip(h.upsample_rates, h.upsample_kernel_sizes)):
            self.ups.append(weight_norm(ConvTranspose1d(h.upsample_initial_channel//(2**i), h.upsample_initial_channel//(2**(i+1)), k, u, padding=(k-u)//2)))
        self.resblocks = nn.ModuleList()
        for i in range(len(self.ups)):
            ch = h.upsample_initial_channel//(2**(i+1))
            for j, (k, d) in enumerate(zip(h.resblock_kernel_sizes, h.resblock_dilation_sizes)):
                self.resblocks.append(ResBlock1(h, ch, k, d))
        self.conv_post = weight_norm(Conv1d(ch, 1, 7, 1, padding=3))
        self.ups.apply(init_weights)
        self.conv_post.apply(init_weights)

    def forward(self, x):
        x = self.conv_pre(x)
        for i in range(self.num_upsamples):
            x = F.leaky_relu(x, LRELU_SLOPE)
            x = self.ups[i](x)
            xs = None
            for j in range(self.num_kernels):
                xs = self.resblocks[i*self.num_kernels+j](x) if xs is None else xs + self.resblocks[i*self.num_kernels+j](x)
            x = xs / self.num_kernels
        x = F.leaky_relu(x)
        x = self.conv_post(x)
        x = torch.tanh(x)
        # ── 수정: 출력 파형 6000 샘플로 crop/pad ──────────────
        if x.shape[-1] > N_SAMPLES:   x = x[..., :N_SAMPLES]
        elif x.shape[-1] < N_SAMPLES: x = F.pad(x, (0, N_SAMPLES - x.shape[-1]))
        return x


# ── 수정: MPD periods [2,3,5,7,11] → [10,20,50] ───────────────
class DiscriminatorP(torch.nn.Module):
    def __init__(self, period, kernel_size=5, stride=3, use_spectral_norm=False):
        super(DiscriminatorP, self).__init__()
        self.period = period
        norm_f = weight_norm if not use_spectral_norm else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(Conv2d(1,    32,   (kernel_size, 1), (stride, 1), padding=(get_padding(5,1), 0))),
            norm_f(Conv2d(32,   128,  (kernel_size, 1), (stride, 1), padding=(get_padding(5,1), 0))),
            norm_f(Conv2d(128,  512,  (kernel_size, 1), (stride, 1), padding=(get_padding(5,1), 0))),
            norm_f(Conv2d(512,  1024, (kernel_size, 1), (stride, 1), padding=(get_padding(5,1), 0))),
            norm_f(Conv2d(1024, 1024, (kernel_size, 1), 1,           padding=(2, 0))),
        ])
        self.conv_post = norm_f(Conv2d(1024, 1, (3,1), 1, padding=(1,0)))

    def forward(self, x):
        fmap = []
        b, c, t = x.shape
        if t % self.period != 0:
            x = F.pad(x, (0, self.period - (t % self.period)), 'reflect')
            t = x.shape[-1]
        x = x.view(b, c, t // self.period, self.period)
        for l in self.convs:
            x = F.leaky_relu(l(x), LRELU_SLOPE); fmap.append(x)
        x = self.conv_post(x); fmap.append(x)
        return torch.flatten(x, 1, -1), fmap


class MultiPeriodDiscriminator(torch.nn.Module):
    def __init__(self):
        super(MultiPeriodDiscriminator, self).__init__()
        # ── 수정: [2,3,5,7,11] → [10,20,50] ──────────────────
        self.discriminators = nn.ModuleList([DiscriminatorP(10), DiscriminatorP(20), DiscriminatorP(50)])

    def forward(self, y, y_hat):
        y_d_rs, y_d_gs, fmap_rs, fmap_gs = [], [], [], []
        for d in self.discriminators:
            r, fr = d(y); g, fg = d(y_hat)
            y_d_rs.append(r); fmap_rs.append(fr)
            y_d_gs.append(g); fmap_gs.append(fg)
        return y_d_rs, y_d_gs, fmap_rs, fmap_gs


class DiscriminatorS(torch.nn.Module):
    def __init__(self, use_spectral_norm=False):
        super(DiscriminatorS, self).__init__()
        norm_f = weight_norm if not use_spectral_norm else spectral_norm
        # ── 수정: kernel 41→7-11, stride 4→1/2 ───────────────
        self.convs = nn.ModuleList([
            norm_f(Conv1d(1,   128, 15, 1, padding=7)),
            norm_f(Conv1d(128, 128,  7, 2, padding=get_padding(7,1))),
            norm_f(Conv1d(128, 256, 11, 1, padding=get_padding(11,1))),
            norm_f(Conv1d(256, 512,  7, 2, padding=get_padding(7,1))),
            norm_f(Conv1d(512, 512,  7, 1, padding=get_padding(7,1))),
            norm_f(Conv1d(512, 512,  5, 1, padding=get_padding(5,1))),
            norm_f(Conv1d(512, 512,  3, 1, padding=get_padding(3,1))),
        ])
        self.conv_post = norm_f(Conv1d(512, 1, 3, 1, padding=1))

    def forward(self, x):
        fmap = []
        for l in self.convs:
            x = F.leaky_relu(l(x), LRELU_SLOPE); fmap.append(x)
        x = self.conv_post(x); fmap.append(x)
        return torch.flatten(x, 1, -1), fmap


class MultiScaleDiscriminator(torch.nn.Module):
    def __init__(self):
        super(MultiScaleDiscriminator, self).__init__()
        # ── 수정: 3개 → 2개 (¼ sub-discriminator 제거) ────────
        self.discriminators = nn.ModuleList([DiscriminatorS(use_spectral_norm=True), DiscriminatorS()])
        self.meanpools = nn.ModuleList([AvgPool1d(4, 2, padding=2)])

    def forward(self, y, y_hat):
        y_d_rs, y_d_gs, fmap_rs, fmap_gs = [], [], [], []
        for i, d in enumerate(self.discriminators):
            if i != 0:
                y     = self.meanpools[i-1](y)
                y_hat = self.meanpools[i-1](y_hat)
            r, fr = d(y); g, fg = d(y_hat)
            y_d_rs.append(r); fmap_rs.append(fr)
            y_d_gs.append(g); fmap_gs.append(fg)
        return y_d_rs, y_d_gs, fmap_rs, fmap_gs


# ── 원본 유지 ──────────────────────────────────────────────────
def feature_loss(fmap_r, fmap_g):
    loss = 0
    for dr, dg in zip(fmap_r, fmap_g):
        for rl, gl in zip(dr, dg):
            loss += torch.mean(torch.abs(rl - gl))
    return loss * 2

def discriminator_loss(disc_real_outputs, disc_generated_outputs):
    loss, r_losses, g_losses = 0, [], []
    for dr, dg in zip(disc_real_outputs, disc_generated_outputs):
        r_loss = torch.mean((1-dr)**2); g_loss = torch.mean(dg**2)
        loss += (r_loss + g_loss)
        r_losses.append(r_loss.item()); g_losses.append(g_loss.item())
    return loss, r_losses, g_losses

def generator_loss(disc_outputs):
    loss, gen_losses = 0, []
    for dg in disc_outputs:
        l = torch.mean((1-dg)**2); gen_losses.append(l); loss += l
    return loss, gen_losses

# 모델 초기화
with open(config_path) as f:
    h = AttrDict(json.load(f))
G   = Generator(h).to(device)
MPD = MultiPeriodDiscriminator().to(device)
MSD = MultiScaleDiscriminator().to(device)
out = G(spec_b.to(device))
print(f'Generator 입력: {spec_b.shape}')
print(f'Generator 출력: {out.shape}')
print(f'파라미터: {sum(p.numel() for p in G.parameters()):,}')

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Generator 입력: torch.Size([8, 81, 130])
Generator 출력: torch.Size([8, 1, 6000])
파라미터: 3,562,498


In [ ]:
# 셀 8. Loss 함수
# ── 수정: Mel-spectrogram Loss → Linear STFT Loss ───────────────
def stft_loss(y_hat, y):
    window = torch.hann_window(WIN_LENGTH).to(y.device)
    S     = torch.stft(y.squeeze(1),     N_FFT, HOP_LENGTH, WIN_LENGTH, window=window, return_complex=True)
    S_hat = torch.stft(y_hat.squeeze(1), N_FFT, HOP_LENGTH, WIN_LENGTH, window=window, return_complex=True)
    S_mag     = torch.abs(S)     + 1e-8
    S_hat_mag = torch.abs(S_hat) + 1e-8
    L_sc  = torch.norm(S_mag - S_hat_mag, p='fro') / (torch.norm(S_mag, p='fro') + 1e-8)
    L_mag = F.l1_loss(torch.log(S_mag), torch.log(S_hat_mag))
    return L_sc + L_mag

print('✅ Loss 함수 정의 완료')

✅ Loss 함수 정의 완료


In [ ]:
# 셀 9. 옵티마이저 & 스케줄러 (원본 유지)
optim_g = torch.optim.AdamW(G.parameters(), h.learning_rate, betas=[h.adam_b1, h.adam_b2])
optim_d = torch.optim.AdamW(itertools.chain(MSD.parameters(), MPD.parameters()), h.learning_rate, betas=[h.adam_b1, h.adam_b2])
scheduler_g = torch.optim.lr_scheduler.ExponentialLR(optim_g, gamma=h.lr_decay)
scheduler_d = torch.optim.lr_scheduler.ExponentialLR(optim_d, gamma=h.lr_decay)
print('✅ 옵티마이저 정의 완료')

✅ 옵티마이저 정의 완료


In [ ]:
# 셀 10. wandb 초기화
# ── 실험마다 name 수정 ─────────────────────────────────────────
import wandb
from google.colab import userdata

wandb.login(key=userdata.get('WANDB_API_KEY'))
wandb.init(
    project='seismic-hifigan',
    entity='joonoo-kwak-korea-university',
    config={
        'epochs':                   N_EPOCHS,
        'batch_size':               BATCH_SIZE,
        'lr':                       h.learning_rate,
        'upsample_initial_channel': h.upsample_initial_channel,
        'mpd_periods':              [10, 20, 50],
        'msd_subdiscs':             2,
        'lambda_stft':              LAMBDA_STFT,
        'grad_clip':                1.0,
    },
    name='exp-finetune-m4-epoch1to200'
)
print('✅ wandb 연동 완료')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


D_loss,█▆▆▆▇▆▆▅▄▄▅▄▄▃▃▃▂▃▂▂▁▃▃▃▃▄▄▅▅▂▄▆▄
G_loss,█▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▃▂▂▂▂▂▂▂▁▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
D_loss,0.24019
G_loss,86.34662
epoch,73


✅ wandb 연동 완료


In [ ]:
ckpt = torch.load(os.path.join(HIGAN_SAVE, 'hifigan_ep070.pth'), map_location=device)
G.load_state_dict(ckpt['G'])
MPD.load_state_dict(ckpt['MPD'])
MSD.load_state_dict(ckpt['MSD'])
optim_g.load_state_dict(ckpt['optim_g'])
optim_d.load_state_dict(ckpt['optim_d'])
start_epoch = ckpt['epoch'] + 1
print(f'✅ 체크포인트 로드 완료 (epoch={ckpt["epoch"]})')

✅ 체크포인트 로드 완료 (epoch=70)


In [ ]:
# 셀 12. 학습 루프
#start_epoch = 1  # 처음부터 시작할 때만 주석 해제

best_loss = float('inf')
history   = {'G': [], 'D': []}

print(f'학습 시작 — epochs={N_EPOCHS}, batch={BATCH_SIZE}, device={device}')
print('=' * 70)

for epoch in range(start_epoch, N_EPOCHS + 1):
    G.train(); MPD.train(); MSD.train()
    epoch_G, epoch_D = 0.0, 0.0
    t0 = time.time()

    for batch_idx, (x, y) in enumerate(train_loader):
        x = x.to(device)
        y = y.to(device)
        y_hat = G(x)

        # ── Discriminator 업데이트 ─────────────────────────────
        optim_d.zero_grad()
        y_df_hat_r, y_df_hat_g, _, _ = MPD(y, y_hat.detach())
        loss_disc_f, _, _ = discriminator_loss(y_df_hat_r, y_df_hat_g)
        y_ds_hat_r, y_ds_hat_g, _, _ = MSD(y, y_hat.detach())
        loss_disc_s, _, _ = discriminator_loss(y_ds_hat_r, y_ds_hat_g)
        loss_disc_all = loss_disc_f + loss_disc_s
        loss_disc_all.backward()
        torch.nn.utils.clip_grad_norm_(
            list(MPD.parameters()) + list(MSD.parameters()), max_norm=1.0)
        optim_d.step()

        # ── Generator 업데이트 ─────────────────────────────────
        optim_g.zero_grad()
        loss_stft = stft_loss(y_hat, y) * LAMBDA_STFT
        y_df_hat_r, y_df_hat_g, fmap_f_r, fmap_f_g = MPD(y, y_hat)
        y_ds_hat_r, y_ds_hat_g, fmap_s_r, fmap_s_g = MSD(y, y_hat)
        loss_fm_f = feature_loss(fmap_f_r, fmap_f_g)
        loss_fm_s = feature_loss(fmap_s_r, fmap_s_g)
        loss_gen_f, _ = generator_loss(y_df_hat_g)
        loss_gen_s, _ = generator_loss(y_ds_hat_g)
        loss_gen_all = loss_gen_f + loss_gen_s + loss_fm_f + loss_fm_s + loss_stft
        loss_gen_all.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
        optim_g.step()

        epoch_G += loss_gen_all.item()
        epoch_D += loss_disc_all.item()

        # ── 배치 중간 출력 ─────────────────────────────────────
        if batch_idx % 100 == 0:
            print(f'  ep{epoch} [{batch_idx}/{len(train_loader)}] '
                  f'G={loss_gen_all.item():.3f} D={loss_disc_all.item():.3f}')

    scheduler_g.step()
    scheduler_d.step()

    n = len(train_loader)
    epoch_G /= n
    epoch_D /= n
    history['G'].append(epoch_G)
    history['D'].append(epoch_D)
    elapsed = time.time() - t0

    print(f'Epoch {epoch:3d}/{N_EPOCHS} | G={epoch_G:.4f} | D={epoch_D:.4f} | {elapsed:.1f}s')
    wandb.log({'G_loss': epoch_G, 'D_loss': epoch_D, 'epoch': epoch})

    if epoch_G < best_loss:
        best_loss = epoch_G
        torch.save({
            'epoch':   epoch,
            'G':       G.state_dict(),
            'MPD':     MPD.state_dict(),
            'MSD':     MSD.state_dict(),
            'optim_g': optim_g.state_dict(),
            'optim_d': optim_d.state_dict(),
            'G_loss':  best_loss
        }, os.path.join(HIGAN_SAVE, 'hifigan_best.pth'))
        print(f'  ✅ best model 저장 (G={best_loss:.4f})')

    if epoch % 10 == 0:
        torch.save({
            'epoch':   epoch,
            'G':       G.state_dict(),
            'MPD':     MPD.state_dict(),
            'MSD':     MSD.state_dict(),
            'optim_g': optim_g.state_dict(),
            'optim_d': optim_d.state_dict(),
            'G_loss':  epoch_G
        }, os.path.join(HIGAN_SAVE, f'hifigan_ep{epoch:03d}.pth'))

wandb.finish()

학습 시작 — epochs=200, batch=8, device=cuda
  ep71 [0/672] G=88.150 D=0.235
  ep71 [100/672] G=88.646 D=0.231


KeyboardInterrupt: 

In [7]:
import os

# git config
os.system('git config --global user.email "joonoo.kwak@gmail.com"')
os.system('git config --global user.name "kwakjoonoo"')

# 파일 복사
os.system('cp "/content/drive/MyDrive/02_hifigan_train_finetune_m4.ipynb" /content/repo/hifigan/')

# git push
os.chdir('/content/repo')
os.system('git checkout hifigan-dev')
os.system('git add hifigan/02_hifigan_train_finetune_m4.ipynb')
os.system('git commit -m "feat: 02 HiFi-GAN train finetune-m4 notebook 추가"')

from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
os.system(f'git push https://{TOKEN}@github.com/isseul/cose362-k08-seismic-generation.git hifigan-dev')

FileNotFoundError: [Errno 2] No such file or directory: '/content/repo'